# 01 – Data collection

**Goal:** Collect the raw data for the analysis from four sources: two web APIs, one web page (scraping) and one file download.
**Input:** web APIs and web pages · **Output:** `data/raw/` (responses stored unchanged), `data/interim/jolpica_*.parquet`, `data/interim/wikipedia_circuits.parquet`, `reports/logs/collection_fastf1.csv`
**Assessment items:** P1 (data collected by the group via web API and web scraping)

| # | Source | Content | Access | Module |
|---|---|---|---|---|
| 1 | FastF1 (F1 live-timing data) | laps, positions, tyres, track status, weather | Python package / web API | `src/collect_fastf1.py` |
| 2 | Jolpica-F1 | calendar, results, qualifying, driver and circuit IDs | REST API (JSON) | `src/collect_jolpica.py` |
| 3 | Wikipedia «List of Formula One circuits» | circuit type, length, turns | web scraping | `src/scrape_wikipedia.py` |
| 4 | bacinger/f1-circuits | circuit layouts and locations (GeoJSON) | file download (pinned commit) | `src/collect_geo.py` |

All collectors first look for the raw files in `data/raw/`. Re-running this notebook therefore does not call the APIs again; `refresh=True` forces a new download. The pilot covers the seasons 2021 (era E1) and 2022 (era E2).

In [ ]:
import sys
from pathlib import Path

# Make the repository root importable, whether Jupyter starts in the root or in notebooks/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

In [ ]:
import json

from src import collect_fastf1, collect_geo, collect_jolpica, scrape_wikipedia
from src.config import PILOT_SEASONS, RAW_DIR

SEASONS = PILOT_SEASONS
SEASONS

## 1. Jolpica-F1 REST API

The API returns at most 100 records per request, so every endpoint is read page by page (`limit`/`offset`). Each page is saved unchanged as JSON; requests are throttled (at least 0.5 s apart) and retried on HTTP 429/5xx.

In [ ]:
jolpica = collect_jolpica.collect(SEASONS)
{name: df.shape for name, df in jolpica.items()}

In [ ]:
# Structure of one raw API page (pagination metadata + nested race table)
page = json.loads((RAW_DIR / "jolpica" / "2021" / "2021_results_offset00000.json").read_text())
mr = page["MRData"]
print({k: mr[k] for k in ("total", "limit", "offset")})
print("first result keys:", list(mr["RaceTable"]["Races"][0]["Results"][0].keys()))

In [ ]:
jolpica["results"].head()

## 2. FastF1 – lap and timing data

FastF1 reads the official F1 live-timing data. Two practical constraints, both handled in `src/collect_fastf1.py`:

- **Access:** since 2026 the live-timing server answers HTTP 401 without an F1 TV account. FastF1 then falls back to its public mirror for historical sessions. The collector never asks for credentials.
- **Rate limit:** FastF1 limits itself to 500 API calls per hour (about 25 races without cache). When the limit is reached, the collector waits and retries instead of skipping races.

Only main races (session `R`) are collected – no sprints, no testing.

In [ ]:
ff1_log = collect_fastf1.collect(SEASONS)
ff1_log["status"].value_counts()

In [ ]:
ff1_log[["season", "round", "event_name", "event_format", "status",
         "n_laps", "n_weather", "n_track_status", "total_laps"]]

In [ ]:
# One raw lap table as delivered by FastF1
laps_raw = pd.read_parquet(RAW_DIR / "fastf1" / "2021" / "01_laps.parquet")
print(laps_raw.shape)
laps_raw[["Driver", "LapNumber", "Time", "LapTime", "Position", "Compound",
          "TyreLife", "PitInTime", "PitOutTime", "TrackStatus", "IsAccurate"]].head()

## 3. Wikipedia – web scraping

The page is downloaded once with an identifying User-Agent and stored unchanged; parsing works on the stored HTML. The circuit table is located by its column names, footnote markers are removed.

In [ ]:
print(scrape_wikipedia.download())
wiki = scrape_wikipedia.parse()
print(wiki.shape)
wiki.head()

## 4. GeoJSON circuit layouts

The files are taken from a fixed commit of the repository, so every download returns identical data.

In [ ]:
print(collect_geo.download())
layouts = collect_geo.load_layouts()
locations = collect_geo.load_locations()
print(layouts.shape, locations.shape, layouts.crs)
layouts.head(3)

## 5. Summary

In [ ]:
summary = pd.DataFrame([
    ("FastF1", "web API (Python package)", "laps", int(ff1_log["n_laps"].sum())),
    ("FastF1", "web API (Python package)", "weather samples", int(ff1_log["n_weather"].sum())),
    ("Jolpica-F1", "REST API", "races", len(jolpica["races"])),
    ("Jolpica-F1", "REST API", "race results", len(jolpica["results"])),
    ("Jolpica-F1", "REST API", "qualifying results", len(jolpica["qualifying"])),
    ("Wikipedia", "web scraping", "circuits", len(wiki)),
    ("bacinger/f1-circuits", "file download", "circuit layouts", len(layouts)),
], columns=["source", "access", "records", "count"])
summary